In [ ]:
# Cell 1 - Setup
import chi, os, time, datetime
from chi import lease, server, network, context

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# Cell 2 - Names 
suffix       = "proj19"
lease_name   = f"bestshot-k8s-{suffix}"
master_name  = f"node-bestshot-master-{suffix}"
worker1_name = f"node-bestshot-worker1-{suffix}"
worker2_name = f"node-bestshot-worker2-{suffix}"
volume_name  = f"bestshot-vol-{suffix}"

# Cell 3 - Create lease
l = lease.Lease(
    lease_name,
    duration=datetime.timedelta(days=7)
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.large"),
    amount=3
)
l.submit(idempotent=True)
l.show()

# Cell 4 - Launch VMs
flavor = l.get_reserved_flavors()[0].name
master  = server.Server(master_name,  image_name="CC-Ubuntu24.04", flavor_name=flavor)
worker1 = server.Server(worker1_name, image_name="CC-Ubuntu24.04", flavor_name=flavor)
worker2 = server.Server(worker2_name, image_name="CC-Ubuntu24.04", flavor_name=flavor)
master.submit(idempotent=True)
worker1.submit(idempotent=True)
worker2.submit(idempotent=True)
master.wait(); worker1.wait(); worker2.wait()
print("All VMs ready!")

# Cell 5 - Floating IP for master only
master.associate_floating_ip()
master.refresh()
fip = master.get_floating_ip()
print(f"Master floating IP: {fip}")
print(f"SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{fip}")

# Cell 6 - Security groups
for sg_name, port, desc in [
    ("allow-ssh",   22,    "SSH"),
    ("allow-http",  80,    "HTTP"),
    ("allow-30283", 30283, "Immich"),
    ("allow-30500", 30500, "MLflow"),
    ("allow-6443",  6443,  "K8S API"),
]:
    sg = network.SecurityGroup({"name": sg_name, "description": desc})
    sg.add_rule("ingress", "tcp", port)
    sg.submit(idempotent=True)
    master.add_security_group(sg_name)
    worker1.add_security_group(sg_name)
    worker2.add_security_group(sg_name)
    print(f"Added: {sg_name}")

# Cell 7 - Block volume
import openstack
conn = chi.connection()
vol = conn.block_storage.create_volume(name=volume_name, size=20)
for _ in range(20):
    vol = conn.block_storage.get_volume(vol.id)
    if vol.status == "available":
        break
    time.sleep(10)
conn.compute.create_volume_attachment(master.id, volumeId=vol.id)
print(f"Volume {volume_name} attached to master")

# Cell 8 - Summary
master.refresh()
print(f"\n{'='*50}")
print(f"DONE! Master IP: {master.get_floating_ip()}")
print(f"SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{master.get_floating_ip()}")
print(f"{'='*50}")